In [7]:
# ── 1: Imports ───────────────────────────────────────────
import sys
sys.path.append("../Src")

import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import text
from cleaning_functions import load_config
from q1_plastic_sources_functions import get_engine
from q5_where_to_intercept_functions import *

In [8]:
# ── 2: Connect ───────────────────────────────────────────
load_dotenv()
config = load_config()
engine = get_engine(os.getenv("DB_PASSWORD"))

In [9]:
rivers_raw = pd.read_parquet(config["output_data"]["file1"])
df_countries = pd.read_sql("SELECT country_name, income_group, continent_id FROM countries", engine)
rivers_merged = load_rivers(rivers_raw, df_countries)

Rivers loaded: 31,819
       emission  income_score
count  31819.00      31819.00
mean      31.62          2.71
std      416.88          0.79
min        0.00          1.00
25%        0.33          2.00
50%        1.29          3.00
75%        7.24          3.00
max    62591.90          4.00


In [10]:
interceptors_df = build_interceptors()

Total interceptors (incl. ocean system) : 22
River interceptors total                : 21
  — In operation                        : 17
  — Installed for testing               : 2
  — Operation paused                    : 1
  — Maintenance                         : 1
Countries covered                       : 10


In [11]:
rivers_merged = match_interceptors_to_rivers(rivers_merged, interceptors_df)

[003] Manual override → Vietnam
[006] Manual override → Guatemala

Rivers flagged      : 17
Emission covered    : 2.6% of total


In [12]:
rivers = run_hdbscan(rivers_merged)
summary = cluster_summary(rivers)

Cluster distribution:
cluster_hdbscan
-1       33
 0     1040
 1    12704
 2     5460
 3    12582
Name: count, dtype: int64

Noise points (-1): 33
 cluster_hdbscan  n_rivers  avg_emission  total_emission  avg_income_score
              -1        33        4033.5        133106.5               2.3
               0      1040           6.5          6783.2               1.0
               1     12704          43.6        554162.1               2.0
               2      5460           3.6         19383.8               4.0
               3     12582          23.3        292548.5               3.0


In [13]:
coverage = compute_coverage(rivers)

Total emission    : 1,005,984 t/yr
Covered           : 25,816 t/yr (2.6%)
Uncovered         : 980,168 t/yr (97.4%)


In [14]:
top5 = top5_uncovered_rivers(rivers)

 rank     country  emission  cumulative_share_pct
    1 Philippines   62591.9              6.221957
    2 Philippines   13450.2              7.558976
    3       India   13432.9              8.894275
    4 Philippines   12398.3             10.126730
    5 Philippines    9339.9             11.055164


In [15]:
plot_interceptor_map(rivers, interceptors_df).show()
plot_cluster_map(rivers).show()
plot_top5_uncovered(top5, rivers).show()
plot_high_income_bubble(rivers).show()

In [16]:
interceptors = build_interceptors()
print(interceptors[['interceptor_id', 'river', 'country', 'status']].to_string())

Total interceptors (incl. ocean system) : 22
River interceptors total                : 21
  — In operation                        : 17
  — Installed for testing               : 2
  — Operation paused                    : 1
  — Maintenance                         : 1
Countries covered                       : 10
   interceptor_id                 river             country                    status
0             001      Cengkareng Drain           Indonesia              In operation
1             002           Klang River            Malaysia              In operation
2             005           Klang River            Malaysia               Maintenance
3             003         Can Tho River             Vietnam              In operation
4             004             Rio Ozama  Dominican Republic          Operation paused
5             007         Ballona Creek       United States              In operation
6             008    Kingston Pen Gully             Jamaica              In operation


In [17]:
interceptors = build_interceptors()
interceptors.to_parquet("../Data/Clean/interceptors.parquet", index=False)
print("✅ interceptors.parquet updated")
print(interceptors.columns.tolist())

Total interceptors (incl. ocean system) : 22
River interceptors total                : 21
  — In operation                        : 17
  — Installed for testing               : 2
  — Operation paused                    : 1
  — Maintenance                         : 1
Countries covered                       : 10
✅ interceptors.parquet updated
['interceptor_id', 'river', 'city', 'country', 'lat', 'lon', 'year_deployed', 'type', 'status', 'has_interceptor']
